# Olist E-commerce Exploratory Analysis

This notebook reproduces the portfolio findings from the processed Olist tables. It keeps raw files unchanged, aggregates transaction tables at the correct grain, and uses descriptive language for associations rather than causal claims.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'notebooks':
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / 'data' / 'processed'
ASSETS_DIR = PROJECT_DIR / 'assets'
ASSETS_DIR.mkdir(exist_ok=True)

NAVY = '#17324D'
BLUE = '#2878B5'
LIME = '#B8D95D'
RED = '#D94F5C'
MUTED = '#68747A'
BRL_MILLIONS = FuncFormatter(lambda value, _: f'R${value / 1_000_000:.1f}M')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Load processed tables

In [ ]:
orders = pd.read_csv(
    DATA_DIR / 'olist_orders.csv',
    parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'],
)
customers = pd.read_csv(DATA_DIR / 'olist_customers.csv', dtype={'customer_zip_code_prefix': 'string'})
items = pd.read_csv(DATA_DIR / 'olist_order_items.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'olist_order_reviews.csv')
products = pd.read_csv(DATA_DIR / 'olist_products.csv')

table_inventory = pd.DataFrame({
    'table': ['orders', 'customers', 'order_items', 'payments', 'reviews', 'products'],
    'rows': [len(orders), len(customers), len(items), len(payments), len(reviews), len(products)],
})
table_inventory

## 2. Build one-row-per-order analysis table

In [ ]:
item_metrics = items.groupby('order_id', as_index=False).agg(
    item_count=('order_item_id', 'count'),
    product_value=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    order_value=('item_total_value', 'sum'),
)
payment_metrics = payments.groupby('order_id', as_index=False).agg(
    payment_value=('payment_value', 'sum')
)
review_metrics = reviews.groupby('order_id', as_index=False).agg(
    average_review_score=('review_score', 'mean')
)

order_summary = (
    orders
    .merge(customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='left', validate='one_to_one')
    .merge(item_metrics, on='order_id', how='left', validate='one_to_one')
    .merge(payment_metrics, on='order_id', how='left', validate='one_to_one')
    .merge(review_metrics, on='order_id', how='left', validate='one_to_one')
)

delivered = order_summary.loc[order_summary['order_status'].eq('delivered')].copy()

customer_orders = delivered.groupby('customer_unique_id')['order_id'].nunique()
summary_kpis = pd.Series({
    'Delivered orders': len(delivered),
    'Unique customers': delivered['customer_unique_id'].nunique(),
    'Product value (BRL)': delivered['product_value'].sum(),
    'Delivered GMV (BRL)': delivered['payment_value'].sum(),
    'Average order value (BRL)': delivered['payment_value'].mean(),
    'Late-delivery rate (%)': delivered['is_late_delivery'].mean() * 100,
    'Average review score': delivered['average_review_score'].mean(),
    'Repeat-customer rate (%)': customer_orders.gt(1).mean() * 100,
})
summary_kpis

## 3. Monthly delivered GMV

In [ ]:
monthly = (
    delivered.assign(purchase_month=delivered['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp())
    .query("purchase_month >= '2017-01-01' and purchase_month <= '2018-08-31'")
    .groupby('purchase_month', as_index=False)
    .agg(gmv_brl=('payment_value', 'sum'), orders=('order_id', 'count'))
)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(monthly['purchase_month'], monthly['gmv_brl'], color=BLUE, linewidth=2.5, marker='o', markersize=4)
ax.fill_between(monthly['purchase_month'], monthly['gmv_brl'], color=BLUE, alpha=0.10)
ax.set(title='Delivered GMV accelerated through 2017 and remained above R$0.9M per month in 2018', xlabel='', ylabel='Delivered GMV (BRL)')
ax.yaxis.set_major_formatter(BRL_MILLIONS)
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'monthly_gmv.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Top product categories

In [ ]:
item_detail = (
    items
    .merge(orders[['order_id', 'order_status']], on='order_id', how='inner', validate='many_to_one')
    .merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left', validate='many_to_one')
)
top_categories = (
    item_detail.loc[item_detail['order_status'].eq('delivered')]
    .assign(product_category=lambda data: data['product_category_name_english'].fillna('unknown'))
    .groupby('product_category', as_index=False)['price'].sum()
    .nlargest(10, 'price')
    .sort_values('price')
)
top_categories['category_label'] = top_categories['product_category'].str.replace('_', ' ').str.title()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_categories['category_label'], top_categories['price'], color=BLUE)
ax.set(title='Top 10 product categories by delivered product value', xlabel='Product value (BRL)', ylabel='')
ax.xaxis.set_major_formatter(BRL_MILLIONS)
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'top_categories_product_value.png', dpi=180, bbox_inches='tight')
plt.show()

## 5. Delivery performance and review score

In [ ]:
delivery_review = (
    delivered.dropna(subset=['is_late_delivery'])
    .assign(delivery_group=lambda data: data['is_late_delivery'].map({0: 'On time or early', 1: 'Late'}))
    .groupby('delivery_group', as_index=False)
    .agg(
        orders=('order_id', 'count'),
        average_delivery_days=('delivery_days', 'mean'),
        average_review_score=('average_review_score', 'mean'),
    )
)

ordered_groups = ['On time or early', 'Late']
delivery_review['delivery_group'] = pd.Categorical(delivery_review['delivery_group'], ordered_groups, ordered=True)
delivery_review = delivery_review.sort_values('delivery_group')

fig, ax = plt.subplots(figsize=(8, 5.5))
colors = [BLUE, RED]
bars = ax.bar(delivery_review['delivery_group'].astype(str), delivery_review['average_review_score'], color=colors, width=0.58)
ax.bar_label(bars, labels=[f'{value:.2f}' for value in delivery_review['average_review_score']], padding=4, fontweight='bold')
ax.set(title='Late delivery is associated with materially lower review scores', xlabel='', ylabel='Average review score', ylim=(0, 5))
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'delivery_review.png', dpi=180, bbox_inches='tight')
plt.show()

delivery_review

## 6. Geographic concentration

In [ ]:
top_states = (
    delivered.groupby('customer_state', as_index=False)['payment_value'].sum()
    .nlargest(10, 'payment_value')
    .sort_values('payment_value')
)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(top_states['customer_state'], top_states['payment_value'], color=NAVY)
ax.set(title='São Paulo dominates delivered GMV', xlabel='Delivered GMV (BRL)', ylabel='Customer state')
ax.xaxis.set_major_formatter(BRL_MILLIONS)
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'top_states_gmv.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. First-order delivery and 90-day repeat purchase


In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))
from src.delivery_analysis import build_facts
from src.write_delivery_report import wilson

snapshot_orders, cohorts = build_facts(DATA_DIR)
eligible = cohorts.loc[cohorts['eligible']]
repeat_groups = eligible.groupby('first_delivery_status').agg(customers=('customer_unique_id', 'count'), repeats=('repeat_within_window', 'sum'))
repeat_groups['rate'] = repeat_groups['repeats'] / repeat_groups['customers'] * 100
fig, ax = plt.subplots(figsize=(8, 4.5))
positions = ['on_time', 'late']
values = [repeat_groups.loc[key, 'rate'] for key in positions]
ax.bar(['First order on time', 'First order late'], values, color=[BLUE, NAVY])
for index, key in enumerate(positions):
    row = repeat_groups.loc[key]
    low, high = wilson(int(row['repeats']), int(row['customers']))
    ax.errorbar(index, values[index], yerr=[[values[index]-low*100], [high*100-values[index]]], fmt='none', ecolor=NAVY, capsize=5)
ax.set(title='Repeat purchase within 90 days', ylabel='Eligible customers repeating (%)', ylim=(0, max(values)*1.6))
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
fig.savefig(ASSETS_DIR / 'first_order_delivery_retention.png', dpi=160)
plt.show()
repeat_groups


## 8. Conclusions

- Delivered GMV reached **R$15.42M** with an average order value of **R$159.85**.
- Jan–Aug 2018 delivered GMV was **143.33%** above the comparable 2017 period.
- Late orders averaged **2.27/5** in reviews versus **4.29/5** for on-time or early orders.
- São Paulo generated approximately **37.4%** of delivered GMV.
- The repeat-customer rate was approximately **3.0%**.
- Customers whose first delivered order was late repeated at **2.52%**, compared with **3.04%** following an on-time or early first order.

These findings are descriptive associations. Additional operational and customer data would be required for causal conclusions.